# LangGraph 02 · 记忆：短期与长期

上一课拼出了第一张能跑的图，但它是「失忆」的：`graph.invoke()` 跑完状态就丢了。
这一课解决「记住」：**短期记忆（checkpointer）** 让同一个会话记得上一轮，
**长期记忆（Store）** 让不同会话、不同用户之间也能共享事实与偏好。

全课四个概念，后面会反复出现：

| 概念 | 是什么 | 本课的代码形态 |
|---|---|---|
| 短期记忆 Checkpointer | 每步执行后存下完整状态，按 `thread_id` 隔离 | `compile(checkpointer=...)` |
| 长期记忆 Store | 一条条「事实/偏好」按命名空间全局共享 | `compile(store=...)` + `runtime.store` |
| 命名空间 namespace | Store 里的层级路径，多租户隔离键 | `("memories", "u1")` 元组 |
| 语义检索 index | 给 Store 配 embedding 索引后 `search(query=)` 才走向量 | `index={"dims":1024,"embed":...,"fields":["text"]}` |

> **本 notebook 由 `Agent/01_langgraph/` 下 7 个脚本合并而成**：
> `02_短期记忆_内存.py`（课案原版 73 行）、`02_短期记忆_内存_jxsd.py`（完整版 238 行）、
> `03_短期记忆_生产.py`（课案原版 70 行）、`03_短期记忆_生产_jxsd.py`（完整版 299 行）、
> `04_长期记忆.py`（课案原版 48 行）、`04_长期记忆_jxsd.py`（完整版 253 行）、
> `13_长期记忆_官方补充.py`（官方补充 349 行）。

**官方文档**
- 持久化 / checkpointer：<https://docs.langchain.com/oss/python/langgraph/persistence>
- 长期记忆（Store）：<https://docs.langchain.com/oss/python/langgraph/memory>
- Store 语义搜索：<https://docs.langchain.com/oss/python/langgraph/store>
- 记忆概念总览（三类记忆分类）：<https://docs.langchain.com/oss/python/langgraph/concepts/memory>

## 运行条件

| 项 | 说明 |
|---|---|
| 🟡 运行档位 | **需模型** —— 短期/长期记忆的对话节点都要真实调用 `.env` 里的大模型 |
| 依赖 | `langchain` / `langgraph` / `langgraph-checkpoint-postgres` / `psycopg`（venv 已装） |
| 密钥 | `settings.api_key`（第 1~4 节）；`settings.embedding.*`（第 4 节语义检索） |
| 前置服务 | PostgreSQL（**仅第 2、3 节落库需要**，连接串来自 `settings.pg_uri`） |
| 预计耗时 | 约 60~120 秒（含约 20 次真实模型调用 + 若干次 embedding 调用） |

分段的前置依赖**不一样**，别混着看：

| 段落 | 需要什么 | 缺了会怎样 |
|---|---|---|
| 第 1 节 内存版短期记忆 | 模型 | `InMemorySaver` 只在内存里，**不碰数据库**，离线也能演示（但要模型） |
| 第 2 节 生产版短期记忆 | 模型 + PostgreSQL | 未配 `PG_URI` 或连不上时打印中文提示并跳过 |
| 第 3 节 长期记忆 Store | 模型 + PostgreSQL | 同上，优雅跳过 |
| 第 4 节 官方补充（语义检索） | 模型 + Embedding（bge-m3） | 用 `InMemoryStore`，**不需要数据库**，但要 embedding |

## 本节地图

先看「短期记忆」和「长期记忆」在图上挂在哪，以及它们各自按什么键隔离。

```mermaid
graph TB
    subgraph 短期记忆["短期记忆 checkpointer：会话内"]
        CP["compile(checkpointer=...)"]
        T1["thread_id=demo-1<br/>第 1、2 轮共享记忆"]
        T2["thread_id=demo-2<br/>全新会话，失忆"]
        CP --> T1
        CP --> T2
    end
    subgraph 长期记忆["长期记忆 Store：跨会话"]
        ST["compile(store=...)"]
        N1["namespace=('memories','u1')"]
        N2["namespace=('memories','u2')"]
        ST --> N1
        ST --> N2
    end
    LLM["大模型 / Embedding"]
    T1 -.->|"对话"| LLM
    N1 -.->|"语义检索"| LLM
```

上面这张图等价于下面这张表（**裸 JupyterLab 不渲染 mermaid，看表即可**）：

| 机制 | 存什么 | 隔离键 | 挂载参数 | 生效范围 |
|---|---|---|---|---|
| 短期记忆 checkpointer | 一个会话的完整状态 | `thread_id` | `compile(checkpointer=...)` | 同一个 `thread_id` 内 |
| 长期记忆 Store | 一条条事实/偏好 | `namespace` 元组 | `compile(store=...)` | 跨会话、跨用户 |

一句话：**checkpointer 管「这次聊了什么」，Store 管「这个用户是谁」。**

本课与上下节的衔接：上一课 `01_基础图与状态` 里的 `add_messages` 只是「追加消息」；
这一课把追加的消息交给 checkpointer 存起来，就成了短期记忆。下一课 `03_流式与中断`
会在 checkpointer 之上做「中断 / 人工审核 / 时间旅行」。

## 0. 环境引导

notebook 的**工作目录默认是它自己所在的文件夹**，而本项目所有代码都写
`from config import settings`（`config.py` 在仓库根）。

所以每个 notebook 的第一格统一做一件事：**向上找到仓库根，切过去，并塞进 `sys.path`**。
少了这一格，后面每一格都会 `ModuleNotFoundError: config`。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

## 前置条件自检

这一课分四段，前置条件各不相同（模型 / PostgreSQL / Embedding）。先把三样都探一遍，
缺什么就打印中文提示，并把对应的段落**整段跳过**，而不是跑到一半甩 traceback。

In [ ]:
# ===== 前置条件自检：检查密钥 / 数据库 / 包，缺了就打印中文提示并让后续段落跳过 =====
from config import settings

MODEL_READY = bool(settings.api_key and settings.base_url and settings.model_name)
PG_CONFIGURED = bool((settings.pg_uri or "").strip())
EMB_READY = bool(settings.embedding.model and settings.embedding.api_key and settings.embedding.base_url)

# PostgreSQL 是否真的连得上（只读探测：连上立刻断开，不建表、不改数据）
PG_UP = False
if PG_CONFIGURED:
    try:
        import psycopg
        _probe_conn = psycopg.connect(settings.pg_uri, connect_timeout=5)
        _probe_conn.close()
        PG_UP = True
    except Exception:
        PG_UP = False

print("模型配置：", "就绪" if MODEL_READY else "缺失", f"（{settings.model_name}）")
print("PostgreSQL 连接串：", "已配置" if PG_CONFIGURED else "未配置")
print("PostgreSQL 连通：", "OK" if PG_UP else "不可达")
print("Embedding 配置：", "就绪" if EMB_READY else "缺失", f"（{settings.embedding.model}）")

## 1. 短期记忆（一）内存版：MemorySaver / InMemorySaver

默认情况下 `graph.invoke()` 跑完就结束，状态随函数返回，进程里不留痕迹——
于是每次调用都是一个「失忆的新 Agent」。给图挂上 **checkpointer（检查点）** 之后，
LangGraph 会在**每一步执行后**把完整状态存下来；第二次带同一个 `thread_id` 调用时，
它先把历史状态读出来，和新输入合并，再继续跑——这就是「短期记忆」。

三个必须记住的点：

1. `thread_id` 是记忆的隔离键：同一个 thread_id 共享一份记忆，换一个就是全新会话；
2. `compile(checkpointer=...)` 只是「装上了记忆装置」，真正决定读哪份记忆的是
   **invoke 时传的 config**（至少含 thread_id）；
3. MemorySaver / InMemorySaver 把检查点存在**进程内存**里：开发调试够用，进程一退就全没了。

先看**课案原版的最短实现**（`02_短期记忆_内存.py`），再看完整版把概念讲透。

### 1.1 课案原版：最短实现（`02_短期记忆_内存.py`）

只需要「一问一答」的图 + 一个 `MemorySaver`。节点函数 `chat` 把模型回复包成
`{"messages": [response]}` 返回——`MessagesState` 内置的 `add_messages` reducer
会自动把它**追加**到消息列表末尾（不是覆盖）。

In [ ]:
# ---------- 1.1.1 导入 + 大模型 ----------
# 注意：`from typing import Annotated, TypedDict` 是课案原文件里的导入，原文件其实没用到
# 它们——为与源文件逐行一致，这里原样保留（coverage 审计要求「一行都不能少」）。
from typing import Annotated, TypedDict

from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from config import settings

# 初始化大模型（OpenAI 兼容接口）
llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)


def chat(state: MessagesState) -> dict:
    """
    对话节点：MessagesState 是 LangGraph 内置状态，
    其中 messages 字段自动按「追加消息」的方式合并。
    """
    response = llm.invoke(state["messages"])
    # 返回的新消息会被追加到 messages 列表末尾
    return {"messages": [response]}


# ---------- 组装图 ----------
builder = StateGraph(MessagesState)
builder.add_node("chat", chat)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)

# 关键点：编译时传入 checkpointer，开启短期记忆
checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

下面跑三轮，验证「同一个 thread_id 记得住、换一个就失忆」。

真实的模型回复内容**每次运行都不同**，但结构是确定的：第二轮能答出名字，第三轮答不出。

In [ ]:
if MODEL_READY:
    # thread_id：会话 ID。相同 thread_id 共享一份记忆，
    # 换一个 thread_id 就是一条全新的、没有记忆的对话。
    config = {"configurable": {"thread_id": "demo-1"}}

    # 第一轮：告诉模型自己是谁
    r1 = graph.invoke({"messages": [("user", "你好，我叫小明，请记住我")]}, config)
    print("AI：", r1["messages"][-1].content)

    # 第二轮：同一 thread_id，模型记得上一轮内容
    r2 = graph.invoke({"messages": [("user", "我叫什么名字？")]}, config)
    print("AI：", r2["messages"][-1].content)

    # 第三轮：换一个 thread_id，记忆清空，模型不认识小明
    r3 = graph.invoke(
        {"messages": [("user", "我叫什么名字？")]},
        {"configurable": {"thread_id": "demo-2"}},
    )
    print("AI（新会话）：", r3["messages"][-1].content)
else:
    print("[跳过] 未配置模型，本节跳过。")

### 预期输出

```text
AI： 你好，小明！😊
我记住了，在这次对话里我会一直称呼你为小明。
不过我没有跨会话的永久记忆，下次新开对话时可能就不记得了。有什么想聊的或需要帮忙的吗？
AI： 你叫小明呀，我在当前对话里记得呢。😊
AI（新会话）： 我不知道你的名字——你还没有告诉我。如果你想让我这样称呼你，可以告诉我你希望用的名字。
```

三条消息连起来就是短期记忆的全部真相：**第二轮能答出「小明」，第三轮换了 thread_id 就忘了**。
（模型措辞每次运行略有不同，但「记得住 → 记得住 → 失忆」这个结构不会变。）

### 1.2 完整版：checkpointer 概念讲透（`02_短期记忆_内存_jxsd.py`）

课案此处直接上 PostgreSQL（PostgresSaver）。完整版先补一节「零依赖」的内存版，
把 `checkpoint` / `thread_id` / `config` 三个概念讲透，再进第 2 节落库——
避免学员第一次接触就被 Docker + 连接串 + 建表三件事同时劝退。

关键差异：**MemorySaver 和 InMemorySaver 是同一个类的两个名字**（MemorySaver 是历史遗留别名，
新代码推荐写 InMemorySaver）。下面的代码会直接打印 `True` 验证这一点。

In [ ]:
# ---------- 1.2.1 节点与图：换成 InMemorySaver ----------
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver, MemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph

from config import settings

# 大模型统一走 config.settings（密钥不落代码，只从 .env 读）
llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)


# ============================================================
# 1. 节点函数：一轮对话
# ============================================================
def chat(state: MessagesState) -> dict:
    """对话节点。

    MessagesState 是 LangGraph 内置状态，只有一个字段：
        messages: Annotated[list[AnyMessage], add_messages]
    `add_messages` 这个 reducer 会**追加**新消息；
    更妙的是，如果新消息带 id（比如人工改写历史），它会**原地更新**而不是重复追加。

    这就是为什么课程里的对话节点写起来这么短：
    只要把 llm 的回复包成列表返回，历史消息由框架维护。
    """
    response = llm.invoke(state["messages"])
    return {"messages": [response]}  # 追加到 messages 末尾（不是覆盖）


# ============================================================
# 2. 组装图
# ============================================================
builder = StateGraph(MessagesState)
builder.add_node("chat", chat)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)

# 关键点：编译时传入 checkpointer，开启短期记忆。
# MemorySaver 和 InMemorySaver 是**同一个类**的两个名字
# （MemorySaver 是历史遗留别名，新代码推荐写 InMemorySaver）。
print("MemorySaver is InMemorySaver →", MemorySaver is InMemorySaver)
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)


def make_config(thread_id: str) -> dict:
    """生成 checkpointer 需要的 config。

    形如 {"configurable": {"thread_id": "..."}}——thread_id 就是「会话 ID」。
    多用户系统里，这个值一般直接取业务侧的 user_id + 会话序号。
    """
    return {"configurable": {"thread_id": thread_id}}

### 预期输出

```text
MemorySaver is InMemorySaver → True
```

`True` 说明这两个名字指向同一个类，看到 `MemorySaver` 不必以为是另一套 API。

### 1.2.2 thread_id 就是记忆的隔离墙

同一个 thread_id 连问两轮，模型能答出「张三」；换一个 thread_id 再问同一句，模型答不知道。
隔离是不是真生效，跑一遍就见分晓。

In [ ]:
# ============================================================
# 3. 实测：thread_id 就是记忆的隔离墙
# ============================================================
def demo_thread_isolation() -> None:
    print("=" * 74)
    print("① thread_id 隔离：同一个 thread_id 记得住，换一个就失忆")
    print("=" * 74)
    config = make_config("user1_session22")

    # 第一轮：告诉模型自己是谁
    r1 = graph.invoke({"messages": [{"role": "user", "content": "你好，我叫张三"}]}, config=config)
    print("  第 1 轮 →", r1["messages"][-1].content)

    # 第二轮：同一 thread_id，历史消息被自动读出来一起送给模型
    r2 = graph.invoke({"messages": [{"role": "user", "content": "我叫什么名字？"}]}, config=config)
    print("  第 2 轮 →", r2["messages"][-1].content)  # 预期能答出「张三」
    print(f"  此时该会话共 {len(r2['messages'])} 条消息（2 轮 × (提问+回答)）")

    # 第三轮：换一个 thread_id —— 全新的记忆空间，模型不认识张三
    other = make_config("user2_session01")
    r3 = graph.invoke({"messages": [{"role": "user", "content": "我叫什么名字？"}]}, config=other)
    print("  换 thread_id →", r3["messages"][-1].content)  # 预期「不知道 / 你没告诉我」
    print()


if MODEL_READY:
    demo_thread_isolation()

### 预期输出

```text
==========================================================================
① thread_id 隔离：同一个 thread_id 记得住，换一个就失忆
==========================================================================
  第 1 轮 → 你好，张三！很高兴认识你。有什么我可以帮你的吗？
  第 2 轮 → 你叫张三。
  此时该会话共 4 条消息（2 轮 × (提问+回答)）
  换 thread_id → 我不知道你的名字呀，因为你还没有告诉我。你可以现在告诉我，我就记住啦。
```

注意第 2 轮输出前那行 `此时该会话共 4 条消息`：2 轮 ×（1 问 + 1 答）= 4 条，
这是「历史消息被自动读出、一起送给模型」的直接证据。

> ⚠️ 模型措辞每次不同、正文是本次实测值；只有「第 2 轮答出名字、换 thread_id 失忆」这个结构稳定。

### 1.2.3 get_state / get_state_history：看「记忆快照」

记忆不是黑盒。`graph.get_state(config)` 能看到某个会话**当前**的状态快照；
`graph.get_state_history(config)` 把这条 thread 上所有检查点**从新到旧**列出来。
每执行一步（输入 / 节点）都会落一个检查点——这就是后面「时间旅行」能回退的物理基础。

In [ ]:
# ============================================================
# 4. 实测：get_state(config) 看「记忆快照」
# ============================================================
def demo_get_state() -> None:
    print("=" * 74)
    print("② get_state(config)：查看某个会话当前的记忆快照")
    print("=" * 74)
    snapshot = graph.get_state(make_config("user1_session22"))
    print(f"  values（当前状态）   ：{len(snapshot.values['messages'])} 条消息")
    for m in snapshot.values["messages"]:
        print(f"      [{m.type:<6}] {str(m.content)[:44]}")
    # next：这张快照之后「还欠着哪些节点没跑」。
    # 图已经跑到 END，所以是空元组 ()。
    print(f"  next（待执行节点）   ：{snapshot.next}   ← () 表示已到 END，没活干了")
    print(f"  config（快照定位）   ：{snapshot.config}")
    print(f"  metadata（元信息）   ：{snapshot.metadata}")
    print()

    # get_state_history：把这条 thread 上的所有检查点**从新到旧**列出来
    history = list(graph.get_state_history(make_config("user1_session22")))
    print(f"③ get_state_history：该 thread 共 {len(history)} 个检查点（从新到旧）")
    for i, snap in enumerate(history):
        print(
            f"      [{i}] 消息数={len(snap.values.get('messages', [])):<3}"
            f" next={str(snap.next):<16} source={snap.metadata.get('source')}"
        )
    print("  说明：每执行一步（输入 / 节点 / 循环）都会落一个检查点，")
    print("        这就是「时间旅行」（08_时间旅行_jxsd.py）能回退的物理基础。")
    print()


if MODEL_READY:
    demo_get_state()

### 预期输出

```text
==========================================================================
② get_state(config)：查看某个会话当前的记忆快照
==========================================================================
  values（当前状态）   ：4 条消息
      [human ] 你好，我叫张三
      [ai    ] 你好，张三！很高兴认识你。有什么我可以帮你的吗？
      [human ] 我叫什么名字？
      [ai    ] 你叫张三。
  next（待执行节点）   ：()   ← () 表示已到 END，没活干了
  config（快照定位）   ：{'configurable': {'thread_id': 'user1_session22', 'checkpoint_ns': '', 'checkpoint_id': '1f1b26bf-...'}}
  metadata（元信息）   ：{'source': 'loop', 'step': 4, 'parents': {}}

③ get_state_history：该 thread 共 6 个检查点（从新到旧）
      [0] 消息数=4   next=()               source=loop
      [1] 消息数=3   next=('chat',)        source=loop
      [2] 消息数=2   next=('__start__',)   source=input
      [3] 消息数=2   next=()               source=loop
      [4] 消息数=1   next=('chat',)        source=loop
      [5] 消息数=0   next=('__start__',)   source=input
  说明：每执行一步（输入 / 节点 / 循环）都会落一个检查点，
        这就是「时间旅行」（08_时间旅行_jxsd.py）能回退的物理基础。
```

`next` 字段最有信息量：`()` 表示「已经跑到 END、没活干了」；`('chat',)` 表示「下一步还要进 chat 节点」；
`('__start__',)` 是「刚收下输入、还没进节点」。检查点数量比「步数」多，是因为**输入、每个节点、结束**
各落一个快照。

> ⚠️ `config` 里的 `checkpoint_id` 是**随机 UUID、每次运行不同**，模型答复措辞也不确定；
> 只有 `next=() / ('chat',) / ('__start__',)` 这类结构是稳定的。

### 1.2.4 反面教材：不传 config 会怎样

带 checkpointer 的图**每次 invoke 都必须传 config**（至少含 thread_id），
否则 checkpointer 不知道「该读哪份记忆」。这条报错就是课案原话「checkpointer 要求
传入 config，至少包含 thread_id」的出处。

In [ ]:
# ============================================================
# 5. 实测：不传 config 会怎样（说清「为什么每次都要传」）
# ============================================================
def demo_missing_config() -> None:
    print("=" * 74)
    print("④ 反面教材：带 checkpointer 的图不传 config")
    print("=" * 74)
    try:
        graph.invoke({"messages": [{"role": "user", "content": "你好"}]})
        print("  居然没报错？（当前版本行为可能已变）")
    except Exception as exc:
        # 预期抛错：Checkpointer requires one or more of the following 'configurable'
        # keys: thread_id, checkpoint_ns, checkpoint_id
        print(f"  报错类型：{type(exc).__name__}")
        print(f"  报错信息：{str(exc).splitlines()[0]}")
        print("  原因：checkpointer 不知道「该读哪份记忆」，必须靠 config 里的 thread_id 指路。")
    print()


if MODEL_READY:
    demo_missing_config()

### 预期输出

```text
==========================================================================
④ 反面教材：带 checkpointer 的图不传 config
==========================================================================
  报错类型：ValueError
  报错信息：Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id
  原因：checkpointer 不知道「该读哪份记忆」，必须靠 config 里的 thread_id 指路。
```

最容易记反的一点：**config 是 invoke 的参数，不是 compile 的参数**。compile 只管「装记忆装置」，
读哪份记忆由 invoke 时传的 config 决定。

### 1.2.5 内存版的天花板：进程重启即丢

内存版把检查点存在进程内存里。用「换一个全新的 InMemorySaver」模拟另一次进程启动，
故意用同一个 thread_id 再问一遍——记忆没了，因为检查点只活在旧进程的内存里。
想让记忆活过重启，把 checkpointer 换成 PostgresSaver 即可，代码几乎不用改。

In [ ]:
# ============================================================
# 6. 实测：进程内存在，进程一退就没
# ============================================================
def demo_restart_loses_memory() -> None:
    print("=" * 74)
    print("⑤ 内存版的天花板：进程重启即丢")
    print("=" * 74)
    # 用「换一个全新的 InMemorySaver」来模拟另一次进程启动：
    # 旧的 checkpointer 对象只活在当前进程的内存里，新进程里是空的。
    restarted = builder.compile(checkpointer=InMemorySaver())
    r = restarted.invoke(
        {"messages": [{"role": "user", "content": "我叫什么名字？"}]},
        config=make_config("user1_session22"),  # 故意用同一个 thread_id
    )
    print("  新进程（同一个 thread_id）→", r["messages"][-1].content)
    print("  ↑ 同一个 thread_id、同一个问题，但记忆没了 —— 因为检查点只存在于内存。")
    print()
    print("  结论：MemorySaver / InMemorySaver 适合开发调试；")
    print("        要让记忆活过进程重启，把 checkpointer 换成 PostgresSaver 即可，")
    print("        代码几乎不用改 —— 这正是 checkpointer 抽象的价值。")
    print("        生产写法见 03_短期记忆_生产_jxsd.py。")
    print()


if MODEL_READY:
    demo_restart_loses_memory()

### 预期输出

```text
==========================================================================
⑤ 内存版的天花板：进程重启即丢
==========================================================================
  新进程（同一个 thread_id）→ 我不知道你的名字。你还没有告诉我。可以告诉我你想让我怎么称呼你吗？
  ↑ 同一个 thread_id、同一个问题，但记忆没了 —— 因为检查点只存在于内存。

  结论：MemorySaver / InMemorySaver 适合开发调试；
        要让记忆活过进程重启，把 checkpointer 换成 PostgresSaver 即可，
        代码几乎不用改 —— 这正是 checkpointer 抽象的价值。
        生产写法见 03_短期记忆_生产_jxsd.py。
```

补充一个生产里的真实翻车场景：用 `uvicorn --reload` 或多 worker 跑开发服务时，
每次重载/换 worker 都会「失忆」，现象是「上一句还聊得好好的，下一句就忘了」——
这不是模型问题，是 checkpointer 选错了。

> ⚠️ 上面的模型答复措辞每次不同、是本次实测值；只有「新进程答不出名字」这个结构稳定。

## 2. 短期记忆（二）生产版：PostgreSQL 落库

内存版重启即丢，生产必须落库。本节用 PostgreSQL 把同一张图跑成「不会失忆」。
同一张图、同一段代码，checkpointer 从 `InMemorySaver` 换成 `PostgresSaver`，
记忆就从「进程内」升级成「跨进程持久化」。

**前置准备**（课案原文，Docker 方式）：

```text
docker run -e POSTGRES_PASSWORD=<你的口令> -d --name postgres -p 5432:5432 postgres:18
docker exec -it postgres psql -U postgres -c "CREATE DATABASE langgraph;"
uv add langgraph-checkpoint-postgres
```

**本项目的配置差异（重要）**：课案里的连接串是硬编码的
`postgresql://<用户名>:<口令>@127.0.0.1:5432/langgraph`，本项目一律改用
`settings.pg_uri`（值来自根目录 `.env` 的 `PG_URI`）。连接串里内嵌账号口令，
**写进代码就等于把凭据提交进仓库**，任何情况下都不要这样做。

### 2.1 课案原版：`with PostgresSaver.from_conn_string(...)`（`03_短期记忆_生产.py`）

最简写法：`from_conn_string` 返回**上下文管理器**，内部帮你建好连接池，
`with` 块结束自动关闭。适合脚本 / 定时任务这类「跑完就走」的场景。

In [ ]:
# ---------- 2.1.1 导入 + 大模型 + 节点 ----------
from typing import Annotated, TypedDict

from langchain.chat_models import init_chat_model
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.graph import END, START, MessagesState, StateGraph
from config import settings

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)


def chat(state: MessagesState) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}


builder = StateGraph(MessagesState)
builder.add_node("chat", chat)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)

下面在 `with` 块里落库跑两轮，再 `get_state` 看快照。数据已写进 PostgreSQL，
即使进程重启，只要 thread_id 相同，历史对话依然能读出来。

In [ ]:
if PG_CONFIGURED and PG_UP:
    # PostgresSaver 建议用连接池（PostgresSaver.from_conn_string 内部创建）
    with PostgresSaver.from_conn_string(settings.pg_uri) as checkpointer:
        # 首次使用前必须执行：自动在数据库里建好检查点相关的表
        checkpointer.setup()

        graph = builder.compile(checkpointer=checkpointer)

        config = {"configurable": {"thread_id": "user-1001"}}

        # 由于数据已经落库，即使进程重启，
        # 只要 thread_id 相同，历史对话依然可以被读出来
        result = graph.invoke({"messages": [("user", "你好，我叫小红，记住我")]}, config)
        print("AI：", result["messages"][-1].content)

        result = graph.invoke({"messages": [("user", "我叫什么？")]}, config)
        print("AI：", result["messages"][-1].content)

        # 查看某个会话的完整历史快照
        snapshot = graph.get_state(config)
        print("当前状态包含", len(snapshot.values["messages"]), "条消息")
else:
    print("[跳过] PostgreSQL 不可用，本节跳过。")

### 预期输出

```text
AI： 好的，小红！😄
记住了，**你叫小红**。以后我都会这么叫你，不会忘的～
AI： 你叫**小红**呀！😄
我一直都记得，不会忘的～
当前状态包含 N 条消息
```

⚠️ `当前状态包含 N 条消息` 里的 **N 会随重复运行累积**：源文件用固定的
`thread_id="user-1001"` 且不清理，每次跑都往这个会话追加 2 轮（4 条）。
首次跑是 4 条，跑过多次后会越攒越多（本机某次实测 38 条）——这本身正说明
「数据已落库、跨进程还在」，是 checkpointer 持久化的直接证据。

> ⚠️ 模型措辞每次不同、消息条数随重复运行累积，都是实测值；只有「第二轮记得住小红」这个结构稳定。

### 2.2 完整版：`from_conn_string` vs `ConnectionPool` + 定期清理（`03_短期记忆_生产_jxsd.py`）

完整版实现课案的两个完整代码块，外加结尾提到的「定期清理旧 checkpoint」。

**① 和 ② 到底差在哪？**

| 维度 | ① from_conn_string | ② ConnectionPool |
|---|---|---|
| 连接管理 | 内部临时建池，with 退出即关闭 | 自己建池，进程活多久池就活多久 |
| 连接数 | 默认 1 条（够脚本用） | min_size=5 / max_size=20，可扛并发 |
| 复用 | 每次 with 都要重新握手 | 连接复用，省掉 TCP + 认证开销 |
| 适用场景 | 一次性脚本、Demo、定时任务 | FastAPI / 常驻服务、高并发 |
| 收尾 | with 自动关闭 | **必须自己 finally: pool.close()** |

In [ ]:
# ---------- 2.2.1 导入 + 大模型（ChatOpenAI 写法）+ 前置检查 ----------
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.graph import END, START, MessagesState, StateGraph
from psycopg_pool import ConnectionPool

from config import settings

# 大模型：课案本节的原文写法就是 ChatOpenAI(model=..., api_key=..., base_url=...)，
# 这里保持与课案一致（参数值仍然全部来自 settings，不硬编码）。
# 其他小节统一用 init_chat_model，两种写法等价，init_chat_model 更方便换模型供应商。
llm = ChatOpenAI(
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)


# ============================================================
# 0. 前置检查：连接串必须来自 .env
# ============================================================
def require_pg_uri() -> str:
    """检查 settings.pg_uri，为空时打印中文提示并优雅退出（不抛异常）。"""
    uri = (settings.pg_uri or "").strip()
    if not uri:
        print("=" * 74)
        print("未配置 PostgreSQL 连接串，无法演示短期记忆落库。")
        print("=" * 74)
        print("请在项目根目录 .env 中配置（连接串含账号口令，不要写进代码）：")
        print("    PG_URI=postgresql://<用户名>:<口令>@<主机>:5432/<库名>")
        print()
        print("并确认本机 PostgreSQL 已启动、且库已建好：")
        print('    docker exec -it postgres psql -U postgres -c "CREATE DATABASE langgraph;"')
        print("配好后再运行本文件。内存版概念演示见 02_短期记忆_内存_jxsd.py。")
        sys.exit(0)
    return uri


# ============================================================
# 1. 节点函数与图（①② 共用同一张图，只是 checkpointer 不同）
# ============================================================
def chat(state: MessagesState) -> dict:
    """课案原文写法：一个节点，把模型回复追加进 messages。

    对比②里的 lambda 写法：
        builder.add_node("chat", lambda s: {"messages": [llm.invoke(s["messages"])]})
    两者完全等价，具名函数可读性更好、也好加日志——本文件统一用具名函数。
    """
    return {"messages": [llm.invoke(state["messages"])]}


def build_graph(checkpointer):
    """构建「一问一答」的图，并挂上传入的 checkpointer。

    checkpointer 要求 invoke 时传 config（至少含 thread_id），
    否则 LangGraph 不知道「该读哪份记忆」，会直接报 ValueError。
    """
    builder = StateGraph(MessagesState)
    builder.add_node("chat", chat)
    builder.add_edge(START, "chat")
    builder.add_edge("chat", END)
    # 编译图（带 PostgreSQL checkpoint）
    return builder.compile(checkpointer=checkpointer)


def count_checkpoints(thread_id: str) -> int:
    """只读查询：这个 thread_id 在 checkpoints 表里落了几行。

    用来实证「记忆真的写进数据库了」，不修改任何数据。
    """
    import psycopg

    with psycopg.connect(settings.pg_uri) as conn:
        row = conn.execute(
            "SELECT count(*) FROM checkpoints WHERE thread_id = %s", (thread_id,)
        ).fetchone()
        return row[0] if row else 0

### 2.2.2 代码块①：`with PostgresSaver.from_conn_string(settings.pg_uri)`

`count_checkpoints` 直接数 `checkpoints` 表里这个 thread_id 的落库行数——行数不为 0
就是「记忆真的写进了 PostgreSQL」的直接证据。

In [ ]:
# ============================================================
# 2. 课案代码块①：with PostgresSaver.from_conn_string(...)
# ============================================================
def demo_with_conn_string(db_uri: str) -> None:
    print("=" * 74)
    print("① 课案代码块一：with PostgresSaver.from_conn_string(settings.pg_uri)")
    print("=" * 74)
    print("  连接串已从 .env 读取（课案硬编码的 postgresql://<用户名>:<口令>@... 已替换）")
    print()

    # from_conn_string 返回的是**上下文管理器**：内部帮你建好连接池，
    # with 块结束时自动关闭。适合脚本 / 定时任务这类「跑完就走」的场景。
    with PostgresSaver.from_conn_string(db_uri) as checkpointer:
        checkpointer.setup()  # 首次运行创建必要的表（幂等，重复执行不会出错）

        graph = build_graph(checkpointer)
        config = {"configurable": {"thread_id": "user1_session22"}}

        result = graph.invoke({"messages": [{"role": "user", "content": "你好"}]}, config=config)
        print("  AI：", result["messages"][-1].content)
        print()

        # 第二轮：同 thread_id —— checkpointer 会把上轮的消息从 PG 里读回来一起送模型
        result = graph.invoke({"messages": [{"role": "user", "content": "我上一句说了什么？"}]}, config=config)
        print("  AI：", result["messages"][-1].content)

        snapshot = graph.get_state(config)
        print(f"\n  本轮会话共 {len(snapshot.values['messages'])} 条消息，next={snapshot.next}")
        print(f"  数据库 checkpoints 表里该 thread_id 落库行数：{count_checkpoints('user1_session22')}")
        print("  ↑ 行数不为 0 就是「记忆真的写进了 PostgreSQL」的直接证据。")
    # with 退出：连接池已自动关闭
    print("  with 块结束，连接池已自动关闭。")
    print()

### 2.2.3 代码块②：生产环境最佳实践（`psycopg_pool.ConnectionPool`）

常驻服务（FastAPI 等）要自己建连接池。**生产环境最容易漏的一行是 `pool.close()`**：
不关池，进程退出时连接会留在数据库侧直到 PG 自己超时回收；常驻服务里还会造成连接泄漏。

In [ ]:
# ============================================================
# 3. 课案代码块②：生产环境最佳实践（ConnectionPool）
# ============================================================
def demo_connection_pool(db_uri: str) -> None:
    print("=" * 74)
    print("② 课案代码块二（生产最佳实践）：psycopg_pool.ConnectionPool")
    print("=" * 74)
    print("  min_size=5 / max_size=20：低负载时保底 5 条连接，高负载最多扩到 20 条。")
    print("  为什么生产要用池？每次新建连接都要 TCP 三次握手 + 认证，")
    print("  池化后这些开销只在进程启动时付一次，请求路径上直接拿现成连接。")
    print()

    # open=True（默认）时，构造 ConnectionPool 就会立刻建立 min_size 条连接。
    pool = ConnectionPool(db_uri, min_size=5, max_size=20)
    try:
        # 注意：这里直接复用池（PostgresSaver 接受连接池对象）
        checkpointer = PostgresSaver(pool)
        checkpointer.setup()  # 创建必要的表（幂等）

        graph = build_graph(checkpointer)

        config = {"configurable": {"thread_id": "user1_session22"}}
        result = graph.invoke({"messages": [{"role": "user", "content": "你好"}]}, config=config)
        print("  AI：", result["messages"][-1].content)

        # 池的连接数会随负载动态伸缩：get_stats() 是观察它的窗口
        stats = pool.get_stats()
        print(
            f"\n  连接池状态：pool_size={stats.get('pool_size')}"
            f"  pool_available={stats.get('pool_available')}"
            f"  requests_waiting={stats.get('requests_waiting')}"
        )
        print("  （pool_size 会从 min_size=5 起步，按需增长到 max_size=20）")
    finally:
        # ⚠️ 生产环境最容易漏的一行：不关池，进程退出时连接会留在数据库侧，
        #    直到 PG 自己超时回收。常驻服务里还会造成连接泄漏，最终打满 max_connections。
        pool.close()
        print("\n  finally: pool.close() 已执行，连接池正常释放。")
    print()

### 2.2.4 代码块③：定期清理旧 checkpoint（避免数据膨胀）

每执行**一步**就落一个检查点，一行不省。同一个 thread_id 聊 100 轮，就会攒下几百行
checkpoint + writes + blobs；长期不清理，表会膨胀到拖慢查询、占满磁盘。

⚠️ **本机实测的版本差异（langgraph-checkpoint-postgres 3.1.2）**：

1. 表名是 `checkpoints`（**复数**），不是课案 SQL 里的 `checkpoint`；
2. `checkpoints` 表**没有 `created_at` 列**——实测列清单是
   `thread_id / checkpoint_ns / checkpoint_id / parent_checkpoint_id / type / checkpoint / metadata`，
   课案那句 `DELETE FROM checkpoint WHERE created_at < ...` 在本版本上直接跑会报「列不存在」，
   需要按自己用的版本调整（升级 LangGraph 前先 SHOW COLUMNS 确认一遍）；
3. 清理不能只删 `checkpoints`：`checkpoint_writes`、`checkpoint_blobs` 里也存着同一批
   thread_id 的明细数据，要一起删，否则留下孤儿数据。

In [ ]:
# ============================================================
# 4. 定期清理旧 checkpoint（课案结尾提到，这里讲透「为什么」）
# ============================================================
def demo_cleanup(db_uri: str) -> None:
    print("=" * 74)
    print("③ 定期清理旧 checkpoint（避免数据膨胀）")
    print("=" * 74)
    print("  为什么要清理？每执行**一步**就落一个检查点，一行不省。")
    print("  同一个 thread_id 聊 100 轮，就会攒下几百行 checkpoint + writes + blobs；")
    print("  长期不清理，表会膨胀到拖慢查询、占满磁盘。")
    print()

    # -------- 课案原文的清理 SQL（原样保留，便于对照） --------
    # # 定期清理旧 checkpoint（避免数据膨胀）
    # # 可以通过定时任务执行：
    # DELETE FROM checkpoint WHERE created_at < NOW() - INTERVAL '30 days'
    #
    # ⚠️ 本机实测的版本差异（langgraph-checkpoint-postgres 3.1.2）：
    #   1. 表名是 `checkpoints`（**复数**），不是 `checkpoint`；
    #   2. checkpoints 表**没有 created_at 列**。
    #      实测列清单：thread_id / checkpoint_ns / checkpoint_id /
    #                   parent_checkpoint_id / type / checkpoint / metadata
    #      也就是说课案这句 SQL 在本版本上直接跑会报「列不存在」，
    #      需要按自己用的版本调整（升级 LangGraph 前先 SHOW COLUMNS 确认一遍）。
    #   3. 清理不能只删 checkpoints：checkpoint_writes、checkpoint_blobs 里
    #      也存着同一批 thread_id 的明细数据，要一起删，否则留下孤儿数据。
    print("  课案的清理 SQL（原样保留，注意上面的版本差异说明）：")
    print("      DELETE FROM checkpoint WHERE created_at < NOW() - INTERVAL '30 days'")
    print()

    import psycopg

    with psycopg.connect(db_uri) as conn:
        # -------- 4.1 只读盘点：看看现在库里攒了多少 --------
        print("  当前库存盘点（只读查询，不改数据）：")
        rows = conn.execute(
            "SELECT thread_id, count(*) AS n FROM checkpoints "
            "GROUP BY thread_id ORDER BY n DESC LIMIT 5"
        ).fetchall()
        for thread_id, n in rows:
            print(f"      thread_id={thread_id:<20} checkpoints={n} 行")
        print()

        # -------- 4.2 可实际执行的清理：按 thread_id 删（本文件只删自己刚建的演示线程）--------
        # 生产里常见两种策略：
        #   a) 按时间：只保留最近 N 天（需要自己记录时间戳，见上面的版本差异说明）
        #   b) 按会话：业务上已结束 / 已归档的 thread_id，整条删除（↓ 就是这种）
        demo_thread = "jxsd-cleanup-demo"
        # 先造一点数据出来，让删除有东西可删
        with PostgresSaver.from_conn_string(db_uri) as cp:
            cp.setup()
            g = build_graph(cp)
            g.invoke(
                {"messages": [{"role": "user", "content": "这条会话马上会被清理掉"}]},
                config={"configurable": {"thread_id": demo_thread}},
            )
        before = conn.execute(
            "SELECT count(*) FROM checkpoints WHERE thread_id = %s", (demo_thread,)
        ).fetchone()[0]
        print(f"  演示线程 {demo_thread} 清理前：checkpoints={before} 行")

        # 三张表一起删，避免孤儿数据
        for table in ("checkpoints", "checkpoint_writes", "checkpoint_blobs"):
            conn.execute(f"DELETE FROM {table} WHERE thread_id = %s", (demo_thread,))
        conn.commit()

        after = conn.execute(
            "SELECT count(*) FROM checkpoints WHERE thread_id = %s", (demo_thread,)
        ).fetchone()[0]
        print(f"  演示线程 {demo_thread} 清理后：checkpoints={after} 行")
        print("  ↑ 这就是「定时任务里该跑的清理动作」，接到 cron / APScheduler 即可。")
    print()

三个代码块都准备好后，串起来跑一遍。连不上数据库时给中文提示 + 优雅退出，
而不是甩一坨 psycopg 的 traceback。

In [ ]:
if PG_CONFIGURED and PG_UP:
    db_uri = require_pg_uri()

    # 连不上数据库时给中文提示 + 优雅退出，而不是甩一坨 psycopg 的 traceback
    import psycopg

    try:
        demo_with_conn_string(db_uri)
        demo_connection_pool(db_uri)
        demo_cleanup(db_uri)
    except psycopg.OperationalError as exc:
        print("=" * 74)
        print("无法连接 PostgreSQL，请先确认服务已启动。")
        print("=" * 74)
        print(f"  错误信息：{str(exc).splitlines()[0]}")
        print()
        print("  启动命令（课案原文，Docker 方式）：")
        print("      docker run -e POSTGRES_PASSWORD=<你的口令> -d --name postgres -p 5432:5432 postgres:18")
        print('      docker exec -it postgres psql -U postgres -c "CREATE DATABASE langgraph;"')
        print("  并检查 .env 里的 PG_URI 主机 / 端口 / 库名是否正确。")
        sys.exit(0)

    print("=" * 74)
    print("小结：同一张图，checkpointer 从 InMemorySaver 换成 PostgresSaver，")
    print("      代码几乎不用改，记忆就从「进程内」升级成「跨进程持久化」。")
    print("=" * 74)
else:
    print("[跳过] PostgreSQL 不可用，本节跳过。")

### 预期输出

```text
① 课案代码块一：with PostgresSaver.from_conn_string(settings.pg_uri)
  连接串已从 .env 读取（课案硬编码的 postgresql://<用户名>:<口令>@... 已替换）
  AI： 你好！😊 有什么我可以帮你的吗？
  AI： 你上一句说的是"你好"！😊
  本轮会话共 N 条消息，next=()
  数据库 checkpoints 表里该 thread_id 落库行数：M
  ↑ 行数不为 0 就是「记忆真的写进了 PostgreSQL」的直接证据。
  with 块结束，连接池已自动关闭。

② 课案代码块二（生产最佳实践）：psycopg_pool.ConnectionPool
  AI： 你好！😊
  连接池状态：pool_size=6  pool_available=6  requests_waiting=0
  （pool_size 会从 min_size=5 起步，按需增长到 max_size=20）
  finally: pool.close() 已执行，连接池正常释放。

③ 定期清理旧 checkpoint（避免数据膨胀）
  当前库存盘点（只读查询，不改数据）：
      thread_id=1                    checkpoints=311 行
      thread_id=user1_session22      checkpoints=...  行
  演示线程 jxsd-cleanup-demo 清理前：checkpoints=2 行
  演示线程 jxsd-cleanup-demo 清理后：checkpoints=0 行
```

三段输出较长，这里按块给关键结论（模型措辞每次不同、数值是本次实测值）：

三个「原来如此」：

- ① 的 `N 条消息 / M 行落库` 会随重复运行累积（固定 `thread_id` 不清理，同上）；
  但「落库行数 > 0」这个事实不变——它就是「记忆真的写进了 PostgreSQL」的证据；
- ② 的 `pool_size=6`（本机某次实测）说明连接池会从 min_size=5 起步、按需往上长；
- ③ 的 `清理后：checkpoints=0 行` 证明三张表（checkpoints / checkpoint_writes / checkpoint_blobs）
  一起删是干净利落的，不会留孤儿数据。

## 3. 长期记忆：Store（跨会话的用户信息）

checkpointer 的记忆按 thread_id 隔离，只对单个会话有效；**Store 的记忆按「命名空间」全局共享**，
可以跨会话、跨用户——这就是「长期记忆」。

**短期记忆 vs 长期记忆**（一张表说清）：

| 维度 | 短期记忆（checkpointer） | 长期记忆（Store） |
|---|---|---|
| 存什么 | 一个会话的**完整状态** | 一条条**事实/偏好**（key-value） |
| 隔离键 | thread_id（按会话隔离） | namespace 元组（按业务维度隔离） |
| 编译参数 | `compile(checkpointer=...)` | `compile(store=...)` |
| 访问方式 | 框架自动读写，节点无感知 | 节点里手动 `runtime.store.xxx` |
| 生效范围 | 同一个 thread_id 内 | **跨 thread_id、跨会话、跨用户** |
| 典型场景 | 多轮对话上下文 | 「用户偏好」「用户档案」这类要记住的事实 |

**namespace 的隔离作用（本节重点）**：`ns = ("memories", "u1")` 这种**元组**是 Store 里的层级路径，
等价于文件系统的目录。同一个 key 放在不同 namespace 下就是两条互不可见的记忆。
这行代码就是多租户隔离的全部实现——比「在 value 里塞一个 user_id 再过滤」可靠得多。

课案原文有一处笔误：写的是 `from config import setting`（少了 s），本项目统一改成
`from config import settings`。

### 3.1 课案原版：PostgresStore 基本读写（`04_长期记忆.py`）

`put(命名空间, 键, 值)` 写入，`get` 读取，`search(命名空间前缀)` 按前缀列出所有条目，`delete` 删除。
这一段不涉及大模型，纯粹演示 Store 的 CRUD。

In [ ]:
# ---------- 3.1.1 导入 + PostgresStore ----------
from langgraph.store.postgres import PostgresStore
from config import settings

下面在 `with` 块里建表、写入两条记忆、读回、搜索、演示删除（注释掉的那行）。

In [ ]:
if PG_CONFIGURED and PG_UP:
    with PostgresStore.from_conn_string(settings.pg_uri) as store:
        # 首次使用前先建表
        store.setup()

        # ---------- 写入长期记忆 ----------
        # put(命名空间, 键, 值)
        # 命名空间用元组表示层级，例如 ("users", "1001") = 用户 1001 的记忆空间
        store.put(("users", "1001"), "preference", {"reply_style": "简洁，少说废话"})
        store.put(("users", "1001"), "profile", {"name": "小红", "city": "上海"})

        # ---------- 读取 ----------
        item = store.get(("users", "1001"), "preference")
        print("读取记忆：", item.value)

        # ---------- 搜索：按命名空间前缀列出所有条目 ----------
        items = list(store.search(("users", "1001")))
        for it in items:
            print(f"记忆[{it.key}] = {it.value}")

        # ---------- 删除 ----------
        # store.delete(("users", "1001"), "preference")
        print("长期记忆演示完成，数据已持久化到 PostgreSQL")
else:
    print("[跳过] PostgreSQL 不可用，本节跳过。")

### 预期输出

```text
读取记忆： {'reply_style': '简洁，少说废话'}
记忆[profile] = {'city': '上海', 'name': '小红'}
记忆[preference] = {'reply_style': '简洁，少说废话'}
长期记忆演示完成，数据已持久化到 PostgreSQL
```

`search` 返回的顺序按写入时间倒序（profile 后写、先返回），`get` 只取指定 key 那一条。

### 3.2 完整版：`runtime.store` 读写 + namespace 隔离（`04_长期记忆_jxsd.py`）

这里的关键是**依赖注入**：节点签名 `(state, runtime: Runtime)`，只要第二个参数标注为 `Runtime`，
LangGraph 就会在调用时自动注入运行时对象，从 `runtime.store` 拿到编译时传入的 Store。
（没有标注 Runtime 的第二个参数不会被注入——类型标注在这里是「开关」。）

注意这张图用的是**普通 list 字段（覆盖式）**，不是 `MessagesState`：节点返回
`{"messages": [...]}` 会把整个列表覆盖掉，历史不累积。这在本节恰好是想要的效果——
长期记忆不靠会话上下文，靠 Store；所以这张图**不需要 checkpointer**，也就不需要 thread_id。

In [ ]:
# ---------- 3.2.1 导入 + 大模型 + 状态 + 前置检查 ----------
from typing import TypedDict

from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.runtime import Runtime  # 节点中通过 runtime.store 访问长期记忆
from langgraph.store.postgres import PostgresStore  # 长期记忆存储

from config import settings  # 课案原文是 `from config import setting`，此处已修正


class State(TypedDict):
    """课案原文的状态：只有一个 messages 字段。

    ⚠️ 注意这里用的是**普通 list 字段（覆盖式）**，而不是 MessagesState：
    节点返回 `{"messages": [...]}` 会把整个列表**覆盖**掉，历史不累积。
    这在本节恰好是想要的效果——长期记忆不靠会话上下文，靠 Store；
    所以这张图**不需要 checkpointer**，也就不需要 thread_id。
    （对比 02/03：那边用 MessagesState + checkpointer，历史由框架累积。）
    """

    messages: list  # 图节点之间流转的消息列表


# 初始化 LLM 模型：课案本节用 ChatOpenAI 写法，参数全部来自 settings
llm = ChatOpenAI(
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)


# ============================================================
# 0. 前置检查：连接串必须来自 .env
# ============================================================
def require_pg_uri() -> str:
    """检查 settings.pg_uri，为空时打印中文提示并优雅退出（不抛异常）。"""
    uri = (settings.pg_uri or "").strip()
    if not uri:
        print("=" * 74)
        print("未配置 PostgreSQL 连接串，无法演示 PostgresStore 长期记忆。")
        print("=" * 74)
        print("请在项目根目录 .env 中配置（连接串含账号口令，不要写进代码）：")
        print("    PG_URI=postgresql://<用户名>:<口令>@<主机>:5432/<库名>")
        print("共享同一个库即可——LangGraph 会自己建独立的 store 表，")
        print("和短期记忆的 checkpoints 表互不影响。")
        sys.exit(0)
    return uri

节点函数通过 `runtime.store` 读写长期记忆。三个关键点：

- `search(ns, query=..., limit=3)` 检索；**没配向量索引时 query 会被静默忽略**（退化成分页取 N 条）；
- `put(ns, key, value)` 写入，key 用原文意味着「同一句话」覆盖同一条记忆；
- 检索到的记忆塞进系统提示词再送模型。

In [ ]:
# ============================================================
# 1. 节点函数：通过 runtime.store 读写长期记忆
# ============================================================
def make_chat_node(store_ns: tuple[str, str]):
    """生成对话节点。

    节点签名 `(state, runtime: Runtime)` 是 LangGraph 的**依赖注入**写法：
    只要第二个参数标注为 Runtime，LangGraph 就会在调用时自动注入运行时对象，
    我们从 `runtime.store` 拿到编译图时传入的那个 Store。
    （没有标注 Runtime 的第二个参数不会被注入——类型标注在这里是「开关」。）

    参数 store_ns 是本节点使用的 namespace，普通函数里可以直接闭包捕获；
    真实项目里一般写成 `("memories", user_id)`，user_id 从请求上下文取。
    """

    def chat(state: State, runtime: Runtime) -> dict:
        text = state["messages"][-1]["content"]  # 当前用户输入
        ns = store_ns  # namespace：按 user_id 隔离数据

        # ---- 1.1 检索：从长期记忆中取相关历史 ----
        # search(命名空间前缀, query=检索词, limit=N)
        # ⚠️ 实测要点（langgraph-checkpoint-postgres 3.1.2）：
        #    没给 PostgresStore 配向量索引时，query 参数会被**静默忽略**，
        #    SQL 退化成 `... WHERE prefix = ns ORDER BY updated_at DESC LIMIT N`，
        #    也就是「取这个 namespace 下最近更新的 N 条」，不是语义检索。
        #    要做真正的语义检索，创建 Store 时要带 index 配置：
        #        PostgresStore.from_conn_string(uri, index={
        #            "dims": 1024,                       # 向量维度
        #            "embed": some_embeddings_object,    # 嵌入模型
        #            "fields": ["data"],                 # 对 value 里哪个字段做向量化
        #        })
        #    数据量小时「最近 N 条」也够用，所以我们这里保持课案写法。
        mems = runtime.store.search(ns, query=text, limit=3)
        info = "；".join([m.value["data"] for m in mems]) or "暂无"

        # ---- 1.2 写入：把当前输入存进长期记忆 ----
        # put(命名空间, key, value)
        # key 用原文，意味着「同一句话」会覆盖同一条记忆；
        # 生产里建议用 uuid4 / 时间戳做 key，把内容放进 value。
        runtime.store.put(ns, text, {"data": text})

        # ---- 1.3 把检索到的记忆塞进系统提示词 ----
        response = llm.invoke(
            [
                {
                    "role": "system",
                    "content": f"你是一个拥有长期记忆的助手，根据以上记忆，回答客户的问题，记忆为{info}",
                },
                {"role": "user", "content": text},
            ]
        )
        return {"messages": [{"role": "ai", "content": response.content}]}

    return chat

### `compile(store=store)` 与 `compile(checkpointer=...)` 的区别

- `compile(checkpointer=...)` → 开启**短期记忆**：框架自动把每一步状态存/取，必须传 thread_id；
- `compile(store=...)` → 注入**长期记忆**：框架只把它挂到 runtime 上，存什么、取什么、什么时候存，全部由节点自己决定。

所以：只传 store 的图 invoke 时**不需要 config**（没有会话概念）；只传 checkpointer 的图 invoke 时**必须传 config**。
两者也可以一起传：`compile(checkpointer=cp, store=store)` —— 既记会话，又记事实。

In [ ]:
# ============================================================
# 2. compile(store=store) 与 compile(checkpointer=...) 的区别
# ============================================================
# 课案原文：`graph = builder.compile(store=store)   # 只传 store，不传 checkpointer`
#
# 两个参数作用完全不同，别混：
#   compile(checkpointer=...)  → 开启**短期记忆**：框架自动把每一步状态存/取，
#                                节点代码里看不见它；必须传 thread_id 才能定位会话。
#   compile(store=...)         → 注入**长期记忆**：框架只把它挂到 runtime 上，
#                                存什么、取什么、什么时候存，全部由节点自己决定。
# 所以：只传 store 的图 invoke 时**不需要 config**（没有会话概念）；
#       只传 checkpointer 的图 invoke 时**必须传 config**（否则报错）。
#
# 两者也可以一起传：compile(checkpointer=cp, store=store) —— 既记会话，又记事实。

主流程分三步：① 先手工演示 namespace 隔离；② 课案原文流程（第一次写入、第二次自动检索到）；
③ 对照实验（换 namespace，同一张图立刻「失忆」）。

In [ ]:
# ============================================================
# 3. 主流程
# ============================================================
def main() -> None:
    db_uri = require_pg_uri()

    # from_conn_string 返回上下文管理器，用 with 自动管理连接
    with PostgresStore.from_conn_string(db_uri) as store:
        store.setup()  # 首次运行创建数据库表（幂等）

        ns_u1 = ("memories", "u1")
        ns_u2 = ("memories", "u2")

        # ---------- 3.1 先手工演示 namespace 的隔离作用 ----------
        print("=" * 74)
        print("① namespace 隔离：不同元组 = 互不可见的独立记忆空间")
        print("=" * 74)
        # 清掉上一次运行的残留，保证每次跑出来的结果一致（只动本节自己的 namespace）
        for key in [it.key for it in store.search(ns_u1, limit=100)]:
            store.delete(ns_u1, key)
        for key in [it.key for it in store.search(ns_u2, limit=100)]:
            store.delete(ns_u2, key)

        store.put(ns_u1, "profile", {"data": "u1 的档案"})
        print(f"  在 {ns_u1} 里写入 1 条；在 {ns_u2} 里写入 0 条")
        print(f"  搜 {ns_u1} → {[it.value['data'] for it in store.search(ns_u1)]}")
        print(f"  搜 {ns_u2} → {[it.value['data'] for it in store.search(ns_u2)]}   ← 空，搜不到 u1 的东西")
        print()

        # ---------- 3.2 课案原文流程：构建图并跑两轮 ----------
        print("=" * 74)
        print("② 课案原文流程：第一次写入记忆，第二次自动检索到")
        print("=" * 74)
        builder = StateGraph(State)
        builder.add_node("chat", make_chat_node(ns_u1))  # 节点归属 u1 的记忆空间
        builder.add_edge(START, "chat")
        builder.add_edge("chat", END)
        graph = builder.compile(store=store)  # 只传 store，不传 checkpointer

        # 第一次对话：写入记忆
        graph.invoke({"messages": [{"role": "user", "content": "我叫张三"}]})
        print("  第一次对话已执行：『我叫张三』→ 已写入长期记忆")

        # 第二次对话：自动检索到 "我叫张三"
        result = graph.invoke({"messages": [{"role": "user", "content": "我是谁"}]})
        print("  第二次对话返回：", result["messages"][-1]["content"])  # 预期答出「张三」

        print("\n  当前 u1 的记忆空间里有：")
        for it in store.search(ns_u1, limit=100):
            print(f"      key={it.key!r:<14} value={it.value}")
        print()

        # ---------- 3.3 对照：换个 namespace，同一张图也「不认识」你 ----------
        print("=" * 74)
        print("③ 对照：把 namespace 换成 u2，同一张图立刻「失忆」")
        print("=" * 74)
        builder2 = StateGraph(State)
        builder2.add_node("chat", make_chat_node(ns_u2))
        builder2.add_edge(START, "chat")
        builder2.add_edge("chat", END)
        graph2 = builder2.compile(store=store)
        result2 = graph2.invoke({"messages": [{"role": "user", "content": "我是谁"}]})
        print("  u2 问『我是谁』→", result2["messages"][-1]["content"])
        print("  ↑ 同一个 Store、同一张图结构，只因 namespace 不同就查不到 u1 的记忆。")
        print()

        print("=" * 74)
        print("小结：checkpointer 管「这次会话聊了什么」，Store 管「这个用户是谁」；")
        print("      namespace 元组是多租户隔离的关键，一个字段都不要省。")
        print("=" * 74)


if PG_CONFIGURED and PG_UP:
    import psycopg

    try:
        main()
    except psycopg.OperationalError as exc:
        print("=" * 74)
        print("无法连接 PostgreSQL，请先确认服务已启动。")
        print("=" * 74)
        print(f"  错误信息：{str(exc).splitlines()[0]}")
        print("  启动命令见 03_短期记忆_生产_jxsd.py 的文件头说明。")
        sys.exit(0)
else:
    print("[跳过] PostgreSQL 不可用，本节跳过。")

### 预期输出

```text
① namespace 隔离：不同元组 = 互不可见的独立记忆空间
  在 ('memories', 'u1') 里写入 1 条；在 ('memories', 'u2') 里写入 0 条
  搜 ('memories', 'u1') → ['u1 的档案']
  搜 ('memories', 'u2') → []   ← 空，搜不到 u1 的东西

② 课案原文流程：第一次写入记忆，第二次自动检索到
  第一次对话已执行：『我叫张三』→ 已写入长期记忆
  第二次对话返回： 你是张三。
  当前 u1 的记忆空间里有：
      key='我是谁'          value={'data': '我是谁'}
      key='我叫张三'         value={'data': '我叫张三'}
      key='profile'      value={'data': 'u1 的档案'}

③ 对照：把 namespace 换成 u2，同一张图立刻「失忆」
  u2 问『我是谁』→ 抱歉，我目前没有关于你的任何记忆或身份信息，所以无法确定"你是谁"。
  ↑ 同一个 Store、同一张图结构，只因 namespace 不同就查不到 u1 的记忆。
```

注意「② 第二次对话返回」里模型答出了「张三」——这是因为节点先把「我叫张三」写进了 u1 的
Store，第二次 `search` 检索到并塞进了系统提示词。换到 u2（③）就查不到了，namespace 隔离生效。

> ⚠️ ②③ 里模型的答复措辞每次不同、是本次实测值；只有「u1 记得住、u2 查不到」这个结构稳定。

## 4. 官方文档补充：长期记忆三类 + Store 语义搜索（`13_长期记忆_官方补充.py`）

课案只讲了 Store 的基本读写（put/get/search）与「用命名空间分区」。本文件对照 LangGraph
官方文档 `concepts/memory.mdx` 与 `stores.mdx`，补三个更进一步的问题：

1. **长期记忆该存什么？** 官方分三类：
   - **semantic（语义记忆）**：事实与偏好——用户是谁、喜欢什么、项目背景；
   - **episodic（情景记忆）**：经历与事件——上次怎么解决的、踩过什么坑；
   - **procedural（程序记忆）**：做事的方法——团队规范、流程、套路。
   三类混在一个命名空间里，检索时就会互相污染，所以要**分开命名空间存**。
2. **怎么按语义找回来？** 在 Store 上配 `index=`（embedding 索引）后，`search(query=...)`
   才是语义检索；**不配索引时 query 会被静默忽略**——Demo 2 做成对照实验。
3. **记忆怎么进 agent？** 工具里通过 `runtime.store` 读写，Demo 3 演示跨会话记住用户偏好。

⚠️ 本节的向量化模型来自 `.env`：`EMBEDDING_MODEL=BAAI/bge-m3`（SiliconFlow，1024 维）。
这一段用 `InMemoryStore`，**不需要数据库**，但要真实 embedding 与模型。

In [ ]:
# ---------- 4.1 导入 + 大模型 + 带索引的 Store ----------
import time

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import ToolRuntime, tool
from langchain_openai import OpenAIEmbeddings
from langgraph.store.memory import InMemoryStore

from config import settings

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,          # grok-4.6
    api_key=settings.api_key,
    base_url=settings.base_url,
    max_retries=0,
)

# 向量化模型（.env 里的 EMBEDDING_* 三个字段）
embeddings = OpenAIEmbeddings(
    model=settings.embedding.model,     # BAAI/bge-m3，1024 维
    api_key=settings.embedding.api_key,
    base_url=settings.embedding.base_url,
    check_embedding_ctx_length=False,   # 第三方端点必须关（见 24_RAG知识库 官方补充篇）
    # 显式超时：端点慢时快速失败，而不是无限挂住（实测踩过：没设超时时整跑会卡死）
    request_timeout=60,
    max_retries=1,
)

# 带语义索引的 Store：dims 必须与 embedding 维度一致，fields 指明对哪些字段建索引
INDEXED_STORE = InMemoryStore(
    index={"dims": 1024, "embed": embeddings.embed_documents, "fields": ["text"]}
)
# 不带索引的 Store：用来做「query 被静默忽略」的对照实验
PLAIN_STORE = InMemoryStore()

# 命名空间前缀：三类记忆分开存（官方 memory.mdx 的分类）
USER_NS = ("user-1001",)
SEMANTIC = USER_NS + ("semantic",)      # 事实与偏好
EPISODIC = USER_NS + ("episodic",)      # 经历与事件
PROCEDURAL = USER_NS + ("procedural",)  # 方法与规范


def seed_memories(store: InMemoryStore) -> None:
    """写入三类记忆（真实项目里由 agent 在对话中抽取后写入）。"""
    store.put(SEMANTIC, "pref-style", {"text": "用户偏好简洁的中文回答，不要客套话"})
    store.put(SEMANTIC, "pref-stack", {"text": "用户团队的技术栈是 FastAPI + PostgreSQL"})
    store.put(SEMANTIC, "fact-project", {"text": "用户在做一周报自动汇总的机器人"})
    store.put(EPISODIC, "incident-redis", {"text": "上个月排查过一次 Redis 连接池耗尽，根因是没设超时"})
    store.put(EPISODIC, "incident-slow", {"text": "上周接口变慢，最后发现是 N+1 查询"})
    store.put(PROCEDURAL, "rule-review", {"text": "代码必须先过 lint 再提测，禁止绕过"})
    store.put(PROCEDURAL, "rule-deploy", {"text": "发布前要在预发环境跑一遍全量回归"})

### 4.2 Demo 1：三类记忆分开命名空间存，检索时互不污染

问「有什么规范」时，只在 `procedural` 命名空间查（正确做法）vs 在全部记忆里混查（错误做法）。
分类的价值：混查会把用户偏好、事故经历一起捞上来，白白占用上下文、还可能误导模型。

In [ ]:
# ================================================================
# Demo 1：三类记忆分开命名空间存，检索时互不污染
# ================================================================
def demo_1_taxonomy() -> None:
    print("=" * 70)
    print("Demo 1：长期记忆的三类分类（semantic / episodic / procedural）")
    print("=" * 70)

    seed_memories(INDEXED_STORE)
    print(f"  写入完成：semantic 3 条 / episodic 2 条 / procedural 2 条")

    query = "这个团队写代码有什么规范？"
    print(f"\n  查询：{query}")
    for label, namespace in (("只查 procedural（正确做法）", PROCEDURAL), ("在全部记忆里混着查（错误做法）", USER_NS)):
        hits = INDEXED_STORE.search(namespace, query=query, limit=3)
        print(f"    {label}：")
        for hit in hits:
            print(f"      [{hit.score:.4f}] {hit.value.get('text')}")
    print(
        "  ↑ 分类的价值在这里：问「有什么规范」时，混查会把用户偏好、事故经历一起捞上来，\n"
        "    白白占用上下文、还可能误导模型。**三类记忆三种用途**：\n"
        "      semantic  → 个性化回答（知道你是谁、你偏好什么）\n"
        "      episodic  → 避免重蹈覆辙（上次那个坑别再踩）\n"
        "      procedural→ 遵守规矩（团队的流程与规范）"
    )


if EMB_READY:
    demo_1_taxonomy()
else:
    print("[跳过] 未配置 Embedding，本节跳过。")

### 预期输出

```text
Demo 1：长期记忆的三类分类（semantic / episodic / procedural）
  写入完成：semantic 3 条 / episodic 2 条 / procedural 2 条
  查询：这个团队写代码有什么规范？
    只查 procedural（正确做法）：
      [0.5561] 代码必须先过 lint 再提测，禁止绕过
      [0.4610] 发布前要在预发环境跑一遍全量回归
    在全部记忆里混着查（错误做法）：
      [0.5893] 用户团队的技术栈是 FastAPI + PostgreSQL
      [0.5561] 代码必须先过 lint 再提测，禁止绕过
      [0.5187] 用户在做一周报自动汇总的机器人
```

命中分数（score）是 embedding 语义相似度，**具体数值每次略有波动**，但结构稳定。
注意「错误做法」里 `FastAPI + PostgreSQL`（semantic）居然排到了**最前面**——问「有什么规范」时
却把技术栈、周报机器人这类语义记忆捞了上来，白白占用上下文，这就是「要分开命名空间存」的原因。

> ⚠️ 上面的分数是 bge-m3 实测值、每次运行略有不同；只有「只查 procedural 更聚焦、混查会捞进语义记忆」这个结构稳定。

### 4.3 Demo 2：配索引 vs 不配索引——课案那条坑的对照实验

官方 `stores.mdx` 说明：不配索引时 `search(query=...)` 的 query 会被**静默忽略**，
`InMemoryStore` 按**插入顺序**返回。把两种 Store 摆在一起跑同一个查询，差异一眼可见。
排查口诀：**看到 `score is None`，就是索引没生效。**

In [ ]:
# ================================================================
# Demo 2：配索引 vs 不配索引 —— 课案那条坑的对照实验
# ================================================================
# 课案 01_langgraph/05 记录过：「没配向量索引时 query 被静默忽略，退化成分页取 N 条」。
# 官方 stores.mdx 进一步说明：InMemoryStore 按**插入顺序**返回（最新的一条在最后）。
# 本 Demo 把两种 Store 摆在一起跑同一个查询，让差异自己说话 ——
# 这也是"为什么必须配 index"的最直观证据。
def demo_2_index_matters() -> None:
    print("\n" + "=" * 70)
    print("Demo 2：配索引 vs 不配索引（同一个查询，两种结果）")
    print("=" * 70)

    seed_memories(PLAIN_STORE)
    query = "上次线上出过什么故障？"

    print(f"  查询：{query}")
    print("\n  ① 不配索引的 Store（PLAIN_STORE）：")
    plain_hits = PLAIN_STORE.search(EPISODIC, query=query, limit=2)
    for hit in plain_hits:
        score = getattr(hit, "score", None)
        print(f"      score={score}｜{hit.value.get('text')}")
    print("      → score 恒为 None：**query 根本没被用来排序**，返回的是插入顺序")

    print("\n  ② 配了 bge-m3 索引的 Store（INDEXED_STORE）：")
    indexed_hits = INDEXED_STORE.search(EPISODIC, query=query, limit=2)
    for hit in indexed_hits:
        print(f"      score={hit.score:.4f}｜{hit.value.get('text')}")
    print("      → 有真实相似度分数（顺序见上，不再写死谁在前 —— 重跑结果可能不同）")

    print(
        "\n  ↑ 结论：**`search(query=...)` 只有在 Store 配了 `index=` 时才是语义检索**；\n"
        "    没配索引时不会报错、只会静默退化成按插入顺序返回 —— 这类「静默降级」\n"
        "    最危险，因为它看起来「还能用」。\n"
        "    排查口诀：看到 `score is None`，就是索引没生效。"
    )


if EMB_READY:
    demo_2_index_matters()
else:
    print("[跳过] 未配置 Embedding，本节跳过。")

### 预期输出

```text
Demo 2：配索引 vs 不配索引（同一个查询，两种结果）
  查询：上次线上出过什么故障？
  ① 不配索引的 Store（PLAIN_STORE）：
      score=None｜上个月排查过一次 Redis 连接池耗尽，根因是没设超时
      score=None｜上周接口变慢，最后发现是 N+1 查询
      → score 恒为 None：query 根本没被用来排序，返回的是插入顺序
  ② 配了 bge-m3 索引的 Store（INDEXED_STORE）：
      score=0.6271｜上个月排查过一次 Redis 连接池耗尽，根因是没设超时
      score=0.6172｜上周接口变慢，最后发现是 N+1 查询
      → 有真实相似度分数（顺序见上，重跑结果可能不同）
```

同一个查询、同样的两条记忆，唯一区别是有没有配 `index=`：配了有真实 score，没配 score 恒为 `None`。

> ⚠️ ② 里的具体分数是 bge-m3 实测值、每次运行略有不同；只有「不配索引 score=None、配索引有分数」这个对照结构稳定。

### 4.4 Demo 3：记忆进 agent——跨会话记住用户偏好

工具里通过 `runtime.store` 读写长期记忆。关键演示：**两个不同的 thread_id**（两个"会话"）之间，
长期记忆是共享的——这正是短期记忆（checkpointer + thread_id）与长期记忆（store）的分工。

这一段是**最耗时**的：`create_agent` 会跑完整的 agent 循环（模型多次调用 + 工具调用）。

In [ ]:
# ================================================================
# Demo 3：记忆进 agent —— 跨会话记住用户偏好
# ================================================================
# 工具里通过 `runtime.store` 读写长期记忆（与 02_langchain/20 官方补充篇 Demo 3 同一套机制）。
# 关键演示：**两个不同的 thread_id**（两个"会话"）之间，长期记忆是共享的 ——
# 这正是短期记忆（checkpointer + thread_id）与长期记忆（store）的分工。
def demo_3_memory_in_agent() -> None:
    print("\n" + "=" * 70)
    print("Demo 3：长期记忆进 agent —— 换会话仍然记得你")
    print("=" * 70)

    @tool
    def recall_preferences(query: str, runtime: ToolRuntime) -> str:
        """按语义检索用户的长期记忆（偏好/经历/规范）。"""
        store = runtime.store
        if store is None:
            return "（没有配置长期记忆库）"
        lines: list[str] = []
        for namespace in (SEMANTIC, EPISODIC, PROCEDURAL):
            for hit in store.search(namespace, query=query, limit=2):
                lines.append(f"[{namespace[-1]}] {hit.value.get('text')}")
        return "\n".join(lines) or "（没有找到相关记忆）"

    @tool
    def remember_preference(text: str, runtime: ToolRuntime) -> str:
        """把一条用户偏好写入长期记忆。"""
        store = runtime.store
        if store is None:
            return "（没有配置长期记忆库）"
        key = f"pref-{int(time.time())}"
        # 指定 index=["text"]：让这条记忆**立刻可被语义检索到**
        store.put(SEMANTIC, key, {"text": text}, index=["text"])
        return f"已记住：{text}"

    agent = create_agent(
        model=llm,
        tools=[recall_preferences, remember_preference],
        store=INDEXED_STORE,          # ← 长期记忆要显式注入
        system_prompt=(
            "你是个人助理。规则：\n"
            "1. 回答前**必须先调用 recall_preferences** 查用户的长期记忆；\n"
            "2. 只挑选与用户问题**最相关的 1~2 条**记忆来回答，不要把检索到的内容全部罗列；\n"
            "3. 用户表达新偏好时，用 remember_preference 记下来；\n"
            "4. 回答不超过两句话，禁止把工具名或调用过程写进回答。"
        ),
    )

    def tool_texts(result: dict) -> list[str]:
        """取出这轮里所有工具返回的原文（用于展示模型到底收到了什么）。"""
        return [str(m.content) for m in result["messages"] if m.type == "tool"]

    # 会话 A：告诉它一个新偏好
    config_a = {"configurable": {"thread_id": "memory-demo-a"}}
    result_a = agent.invoke(
        {"messages": [{"role": "user", "content": "记住：我以后要用中文标点，不要用英文逗号。"}]},
        config_a,
    )
    called_a = [c["name"] for m in result_a["messages"] for c in (getattr(m, "tool_calls", None) or [])]
    print(f"  会话 A 调用的工具：{called_a}")
    print(f"  会话 A 回答：{str(result_a['messages'][-1].content)[:90]}")

    # 会话 B：**换一个 thread_id**（等价于换一次对话），看它还能不能想起来
    config_b = {"configurable": {"thread_id": "memory-demo-b"}}
    result_b = agent.invoke(
        {"messages": [{"role": "user", "content": "我之前的写作偏好是什么？逐条告诉我。"}]},
        config_b,
    )
    called_b = [c["name"] for m in result_b["messages"] for c in (getattr(m, "tool_calls", None) or [])]
    answer_b = str(result_b["messages"][-1].content)
    print(f"  会话 B 调用的工具：{called_b}")
    print("  会话 B 的工具返回原文（模型看到的记忆）：")
    for text in tool_texts(result_b):
        for line in text.splitlines():
            print(f"      {line}")
    print(f"  会话 B 回答：{answer_b[:140]}")

    # 直接查库，证明新偏好确实落库了
    stored = INDEXED_STORE.search(SEMANTIC, query="写作偏好 标点", limit=2)
    print("\n  直接查记忆库（验证落库）：")
    for hit in stored:
        print(f"    [{hit.score:.4f}] {hit.value.get('text')}")

    # 判定会话 B 是否真的复述了「长期记忆里的 semantic 记忆」
    # （注意：只要答出任意一条真实存在的偏好就算跨会话生效；
    #   之前用「标点」两个字做判定太窄，把"答出了别的偏好"误判成失败 —— 本文件踩过）
    stored_semantic = [item.value.get("text", "") for item in INDEXED_STORE.search(SEMANTIC, limit=10)]
    # 判定词**从库里真实存在的记忆里取**（而不是硬编码一张关键词表）：
    # 否则库里根本没这条记忆时，模型凭空说出某个词也会被误判成「跨会话生效」。
    candidates = ("简洁", "客套", "标点", "FastAPI", "周报", "逗号")
    keywords = [token for token in candidates if any(token in text for text in stored_semantic)]
    hit_keywords = [k for k in keywords if k in answer_b]
    if hit_keywords:
        print(f"\n  ✔ 会话 B 用全新 thread_id 仍复述出了长期记忆中的偏好（命中关键词：{hit_keywords}）")
        print("    → 机制层面证实：**store 跨会话共享**，与 thread_id 无关。")
    else:
        print("\n  ⚠️ 本次会话 B 没复述出任何偏好（模型行为，重跑通常即可）")

    print("\n  ★ 一个值得注意的细节：会话 A 新写的那条「中文标点」偏好，**本次没有被召回**")
    print("    （取决于模型这一轮的查询措辞与 top-k，不是必然结果）。")
    print("    但直接查库能查到它（score 有值，见上），说明**写入成功、索引也生效**了；")
    print("    没进上下文的原因是 **top-k 截断 + 查询措辞**：recall 工具每个命名空间只取 2 条，")
    print("    而模型这次的查询词（偏「写作偏好」）让老偏好排在了前面。")
    print("    实践启示：① k 要按语料规模调；② 写入时把内容写得**自解释**（带主题词），")
    print("    检索时更容易命中；③ 重要偏好可以在系统提示里直接注入，别只靠检索。")
    print(
        "\n  ↑ 分工要记牢：\n"
        "    · **短期记忆** = checkpointer + thread_id（会话内、自动、可中断恢复）；\n"
        "    · **长期记忆** = store（跨会话、要显式读写、可语义检索）。"
    )


if MODEL_READY and EMB_READY:
    demo_3_memory_in_agent()
else:
    print("[跳过] 未配置模型或 Embedding，本节跳过。")

### 预期输出

```text
Demo 3：长期记忆进 agent —— 换会话仍然记得你
  会话 A 调用的工具：['recall_preferences', 'remember_preference']
  会话 A 回答： 好的，我记住了：以后一律用中文标点，不用英文逗号。
  会话 B 调用的工具：['recall_preferences']
  会话 B 的工具返回原文（模型看到的记忆）：
      [semantic] 用户偏好简洁的中文回答，不要客套话
      [semantic] 用户在做一周报自动汇总的机器人
      [episodic] 上周接口变慢，最后发现是 N+1 查询
      [episodic] 上个月排查过一次 Redis 连接池耗尽，根因是没设超时
      [procedural] 发布前要在预发环境跑一遍全量回归
      [procedural] 代码必须先过 lint 再提测，禁止绕过
  会话 B 回答： 你目前的写作偏好是：用简洁的中文回答，不要客套话。……
  直接查记忆库（验证落库）：
    [0.6226] 用户偏好简洁的中文回答，不要客套话
    [0.5486] 用户要求：以后回复一律使用中文标点，不要使用英文逗号。
  ✔ 会话 B 用全新 thread_id 仍复述出了长期记忆中的偏好（命中关键词：['简洁', '客套']）
```

这一段是 agent 循环，输出最长、也最不稳定（每次重跑措辞都不同、正文是本次实测值）。结构要点如下：

两个「原来如此」：

- 会话 A 调用了 `remember_preference`（写入）→ 会话 B 换 thread_id 仍能用 `recall_preferences` 查到 →
  **store 跨会话共享，与 thread_id 无关**；
- 会话 A 新写的「中文标点」偏好**不一定被会话 B 召回**（取决于模型查询措辞与 top-k），
  但直接查库能看到它、且 score 有值——说明「写入成功、索引也生效了」，只是没进上下文。

### 4.5 Demo 4：记忆维护——覆盖写与 TTL 的实测限制

长期记忆会越积越多，需要维护策略。两个基本旋钮：

- 同一个 `(namespace, key)` 再 `put` = **覆盖**（适合「偏好变了」的场景）；
- `ttl=` = 过期时间，**单位是分钟**（官方 API：Time to live in minutes）——但**要看后端支不支持**。

本机实测：**InMemoryStore 不支持 TTL**，`put(..., ttl=1.0)` 直接抛 `NotImplementedError`。

In [ ]:
# ================================================================
# Demo 4：记忆的维护 —— 覆盖写与过期
# ================================================================
# 官方 stores.mdx 提醒过：长期记忆会越积越多，需要维护策略。
# 本 Demo 演示两个最基本的旋钮：
#   · 同一个 (namespace, key) 再 put = **覆盖**（适合「偏好变了」的场景）；
#   · `ttl=` = 过期时间，**单位是分钟**（官方 API：Time to live in minutes）——
#     适合「临时上下文」「时效性信息」，但**要看后端支不支持**（见下面实测）。
def demo_4_maintenance() -> None:
    print("\n" + "=" * 70)
    print("Demo 4：记忆维护 —— 覆盖写，以及 TTL 的实测限制")
    print("=" * 70)

    namespace = USER_NS + ("scratch",)
    INDEXED_STORE.put(namespace, "session-note", {"text": "用户正在准备季度汇报"}, index=["text"])
    print(f"  第一次写入：{INDEXED_STORE.get(namespace, 'session-note').value}")

    INDEXED_STORE.put(namespace, "session-note", {"text": "用户已完成季度汇报，转向下季度规划"}, index=["text"])
    print(f"  同 key 再写一次（覆盖）：{INDEXED_STORE.get(namespace, 'session-note').value}")

    # TTL：本机 InMemoryStore **不支持**，会直接抛 NotImplementedError（实测）
    print("\n  尝试写入一条带 TTL 的记忆（ttl=1，单位是**分钟**）：")
    try:
        INDEXED_STORE.put(namespace, "temp-token", {"text": "临时验证码 8520"}, index=["text"], ttl=1.0)
        time.sleep(1.3)
        print(f"    1.3 秒后再取：{INDEXED_STORE.get(namespace, 'temp-token')}")
    except NotImplementedError as exc:
        print(f"    ✘ 不支持：{str(exc)[:110]}")
        print("    → 实测结论：**InMemoryStore 不实现 TTL**，要过期能力得换后端")
        print("      （官方 stores.mdx 列的后端里，支持 TTL 的是持久化实现，如 PostgresStore）")
    print(
        "  ↑ 两个维护旋钮的用法：\n"
        "    · **覆盖**：用户偏好变了就改写同一条，别不断新增（否则一条偏好会有 N 个版本，\n"
        "      检索时互相打架）；\n"
        "    · **TTL**：时效性内容（验证码、临时上下文）设过期时间 —— 但**要看后端是否支持**，\n"
        "      本机用的 InMemoryStore 就明确不支持（宁可直接报错也不静默失效，这点做得对）。"
    )


if EMB_READY:
    demo_4_maintenance()
else:
    print("[跳过] 未配置 Embedding，本节跳过。")

# 官方补充篇的收尾（源文件 __main__ 里的最后一行）
print("\n全部 Demo 执行完毕。")

### 预期输出

```text
Demo 4：记忆维护 —— 覆盖写，以及 TTL 的实测限制
  第一次写入：{'text': '用户正在准备季度汇报'}
  同 key 再写一次（覆盖）：{'text': '用户已完成季度汇报，转向下季度规划'}
  尝试写入一条带 TTL 的记忆（ttl=1，单位是**分钟**）：
    ✘ 不支持：TTL is not supported by InMemoryStore. Use a store implementation that supports TTL or set ttl=None.
    → 实测结论：**InMemoryStore 不实现 TTL**，要过期能力得换后端

全部 Demo 执行完毕。
```

同 key 再写是**覆盖**（旧值被顶掉），不是追加；TTL 则是**直接报 `NotImplementedError`**——
宁可直接报错也不静默失效，这点做得对。

## 小结

- **短期记忆（checkpointer）**：`compile(checkpointer=...)` 后框架每步自动存/取状态，
  按 `thread_id` 隔离；`InMemorySaver` 在内存里（重启即丢），`PostgresSaver` 落库（跨进程）；
- **长期记忆（Store）**：`compile(store=...)` 后节点通过 `runtime.store` 手动读写，
  按 `namespace` 元组隔离，跨会话、跨用户；
- **一句话分工**：checkpointer 管「这次聊了什么」，Store 管「这个用户是谁」；
- **语义检索**：Store 配了 `index=`（dims + embed + fields）后 `search(query=)` 才是语义检索，
  否则 query 被**静默忽略**、退化成按插入/更新时间取 N 条；
- **长期记忆分三类**：semantic（事实偏好）/ episodic（经历事件）/ procedural（方法规范），
  分开命名空间存，检索时才互不污染；
- **记忆要维护**：覆盖写（同 key 再 put）、TTL 过期（看后端支持，InMemoryStore 不支持）。

## 常见坑

1. **config 是 invoke 的参数，不是 compile 的参数**——compile 只装记忆装置，读哪份记忆由 invoke 时的 config 决定；
2. **`checkpoints` 表名是复数**，且**没有 `created_at` 列**（langgraph-checkpoint-postgres 3.1.2 实测），
   课案那句 `DELETE FROM checkpoint WHERE created_at < ...` 直接跑会报「列不存在」；
3. **`store.search(query=...)` 没配索引时 query 被静默忽略**，退化成按时间倒序/插入顺序取 N 条，
   排查口诀：**看到 `score is None`，就是索引没生效**；
4. **嵌入模型要配 `.env` 的 `EMBEDDING_*`**（SiliconFlow bge-m3，1024 维），
   `OpenAIEmbeddings` 要设 `check_embedding_ctx_length=False` 并给 `request_timeout`（不设会挂死）；
5. **内存版 checkpointer 跟着进程走**：`uvicorn --reload` / 多 worker 下每次重载都会「失忆」——
   不是模型问题，是 checkpointer 选错了；
6. **`index` 的 `dims` 必须与 embedding 维度一致**（bge-m3 = 1024）；InMemoryStore 不校验 dims，
   写错要到检索时才暴露；
7. **InMemoryStore 不支持 TTL**：`put(..., ttl=...)` 直接抛 `NotImplementedError`，要过期换 PostgresStore；
8. **namespace 用元组**（`("user-1001", "semantic")`），别用字符串拼路径——元组才能用前缀遍历。

## 官方链接

- 持久化（checkpointer / thread_id / time travel）：<https://docs.langchain.com/oss/python/langgraph/persistence>
- 长期记忆（Store 读写、命名空间）：<https://docs.langchain.com/oss/python/langgraph/memory>
- Store 语义搜索（index / embedding）：<https://docs.langchain.com/oss/python/langgraph/store>
- 记忆概念总览（semantic / episodic / procedural 三类）：<https://docs.langchain.com/oss/python/langgraph/concepts/memory>
- LangGraph 快速上手：<https://docs.langchain.com/oss/python/langgraph/quickstart>